In [0]:
%skip
#cell 1 - test PySpark is working
df_test = spark.createDataFrame([(1,"test") , (2,"abc")], ["id","value"])
df_test.show()
print("PySpark is working!")

In [0]:
import logging
from datetime import datetime

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

logger.info("Pipeline started")

In [0]:
# Load PPR data from Unity Catalog
df_ppr = spark.read \
    .option("header", True) \
    .option("encoding","CP1252") \
    .csv("/Volumes/workspace/default/ppr/PPR-ALL.csv")

#check it loaded correctly
#print(f"Total records : {df_ppr.count()} ")
#df_ppr.printSchema()
#df_ppr.show(5, truncate=False)

# Clean column names for Bronze layer
df_bronze = df_ppr \
    .withColumnRenamed("Date of Sale (dd/mm/yyyy)", "date_of_sale") \
    .withColumnRenamed("Price (€)", "price_raw") \
    .withColumnRenamed("Not Full Market Price", "not_full_market_price") \
    .withColumnRenamed("VAT Exclusive", "vat_exclusive") \
    .withColumnRenamed("Description of Property", "property_type") \
    .withColumnRenamed("Property Size Description", "property_size") \
    .withColumnRenamed("Address", "address") \
    .withColumnRenamed("County", "county") \
    .withColumnRenamed("Eircode", "eircode")


# Save Bronze - raw data, just column names fixed
df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.ppr_bronze")

logger.info(f"Loaded {df_bronze.count()} records from PPR dataset")
logger.info(f"Bronze table saved with {df_bronze.count()} records")


In [0]:
%skip
# cell - 4 - Avg price by county and year
df_trands = df_clean \
    .groupBy("county", "year") \
    .agg(
        F.round(F.avg("price"), 2).alias("avg_price"),
        F.round(F.median("price"), 2).alias("median_price"),
        F.count("price").alias("total_sales"),
        F.round(F.min("price"), 2).alias("min_price"),
        F.round(F.max("price"), 2).alias("max_price"),
        F.round(F.stddev("price"),2).alias("std_price"),
    ) \
    .withColumn("avg_price", F.format_number("avg_price", 2)) \
    .withColumn("median_price", F.format_number("median_price", 2)) \
    .withColumn("min_price", F.format_number("min_price", 2)) \
    .withColumn("max_price", F.format_number("max_price", 2)) \
    .withColumn("std_price", F.format_number("std_price", 2)) \
    .orderBy("county", "year")

display(df_trands)


In [0]:
%skip
#Save as Delta Table
df_trands.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.ppr_county_trends")

print("Delta table saved")


In [0]:
%skip
# Cell 6 - Dublin price growth story
df_dublin = df_clean \
    .filter(F.col("county") == "Dublin") \
    .groupBy("year") \
    .agg(
        F.round(F.median("price"), 2).alias("median_price"),
        F.count("price").alias("total_sales")
    ) \
    .withColumn("median_price", F.format_number("median_price", 2)) \
    .withColumn("year", F.col("year").cast("int")) \
    .orderBy("year")

display(df_dublin)